---
---
# RETO DEL LAB08 · Completa los tres conjuntos
## Solución comentada

**Curso Big Data e IA Aplicada · Formación San Miguel · Edición Técnica**

Un cuadro de mando necesita **tres formas y no más**: un **titular**, un **ranking** y una
**evolución**. Este cuaderno publica las tres, y explica las dos decisiones que hay dentro.

> ⚠️ **Léelo después de intentarlo.** El reto no era escribir el SQL —ya lo sabes— sino darse
> cuenta de que **publicar no es lo mismo que consultar**, y de que una de las dos consultas
> esconde una decisión que hay que declarar.

## Paso 0 · La vista, como siempre

Se reconstruye LIMPIO-v1 sobre el Parquet maestro. Si no existe todavía, se levanta sobre el CSV
crudo con las tres reglas: **mismos números, otro camino**.

In [ ]:
import duckdb, os

SALIDA = "../datasets/salida"
PARQUET = f"{SALIDA}/ventas_limpio.parquet/*.parquet"

try:
    duckdb.sql(f"CREATE OR REPLACE VIEW ventas_limpio AS "
               f"SELECT * FROM '{PARQUET}'")
    duckdb.sql("SELECT COUNT(*) FROM ventas_limpio").fetchone()
    origen = "el Parquet maestro"
except Exception:
    duckdb.sql("""CREATE OR REPLACE VIEW ventas_limpio AS
        SELECT id_venta, fecha, id_cliente, id_producto, categoria,
               unidades, precio_unitario,
               CASE WHEN COALESCE(TRIM(ciudad), '') = '' THEN NULL
                    ELSE UPPER(SUBSTR(TRIM(ciudad),1,1))
                         || LOWER(SUBSTR(TRIM(ciudad),2)) END AS ciudad,
               canal
        FROM '../datasets/ventas.csv'
        WHERE precio_unitario > 0""")
    origen = "el CSV crudo (plan B)"

filas = duckdb.sql("SELECT COUNT(*) FROM ventas_limpio").fetchone()[0]
print(f"  Origen: {origen}")
print(f"  Filas : {filas}")
print("  ANCLA 999535 -> VALIDADO" if filas == 999535
      else "  NO CUADRA. Revisa antes de publicar nada.")

## Paso 1 · El ranking por ciudad

**La consulta ya la escribiste en el paso 3 del laboratorio.** Aquí no se calcula nada nuevo: se
**publica**. Lo único que cambia respecto al `COPY` del canal es la consulta de dentro.

⚠️ **Fíjate en el `WHERE ciudad IS NOT NULL`.** Es lo que deja fuera las **3.030** ventas sin
ciudad conocida — y por eso salen **10** filas y no 11.

In [ ]:
import os

import duckdb

SALIDA = "../datasets/salida"
os.makedirs(SALIDA, exist_ok=True)

duckdb.sql(f"""COPY (
    SELECT ciudad,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio
    WHERE ciudad IS NOT NULL
    GROUP BY ciudad
    ORDER BY facturacion DESC
) TO '{SALIDA}/kpi_ciudad.csv' (HEADER)""")

print("  Publicado: kpi_ciudad.csv")
duckdb.sql(f"SELECT * FROM '{SALIDA}/kpi_ciudad.csv' LIMIT 4").show()

### La decisión que hay detrás del `IS NOT NULL`

| Si lo pones | Si no lo pones |
|---|---|
| 10 filas · el análisis cubre 996.505 ventas | 11 filas · una de ellas con la ciudad vacía |
| **Se pierden 3.030 ventas del desglose** | No se pierde nada, pero aparece una fila sin nombre |

**Las dos son defendibles.** Lo que no es defendible es no saber cuál has elegido — ni no decirlo
en el informe. Si lo pones, la frase que toca es:

> *«El desglose por ciudad excluye las ventas sin ciudad conocida.»*

## Paso 2 · La evolución mensual

**Esta sí es nueva.** `STRFTIME(fecha, '%Y-%m')` convierte una fecha en la clave del mes.

🎯 **Y ordena bien como texto porque el año va delante.** Con `'%m-%Y'`, enero de 2026 iría antes
que febrero de 2025. **Un formato de fecha no es cosmética.**

In [ ]:
import duckdb

SALIDA = "../datasets/salida"

duckdb.sql(f"""COPY (
    SELECT STRFTIME(fecha, '%Y-%m')                AS mes,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio
    GROUP BY mes
    ORDER BY mes
) TO '{SALIDA}/kpi_mes.csv' (HEADER)""")

print("  Publicado: kpi_mes.csv")
duckdb.sql(f"SELECT * FROM '{SALIDA}/kpi_mes.csv' LIMIT 4").show()

## Paso 3 · ⚓ El control

**Tres canales, diez ciudades, doce meses.** Si alguno no cuadra, algo se ha publicado mal y hay
que arreglarlo **antes** de seguir.

In [ ]:
import duckdb

SALIDA = "../datasets/salida"
ESPERADO = {"kpi_canal.csv": 3, "kpi_ciudad.csv": 10, "kpi_mes.csv": 12}

todo_bien = True
for fichero, filas_esperadas in ESPERADO.items():
    try:
        n = duckdb.sql(f"SELECT COUNT(*) FROM '{SALIDA}/{fichero}'").fetchone()[0]
    except Exception:
        print(f"  {fichero:18s} NO EXISTE")
        todo_bien = False
        continue
    marca = "OK" if n == filas_esperadas else "NO CUADRA"
    print(f"  {fichero:18s} {n:3d} filas (se esperaban {filas_esperadas})  {marca}")
    todo_bien = todo_bien and n == filas_esperadas

print()
print("  LOS TRES CONJUNTOS, PUBLICADOS Y VERIFICADOS" if todo_bien
      else "  REVISA lo que no cuadra antes de seguir")

## Paso 4 · La comprobación cruzada

Lo mismo que hará tu aplicación el viernes: **comparar lo publicado con el cálculo en vivo**.

Son dos caminos que no se conocen entre sí — un `COPY` de hace un rato y una consulta de ahora
mismo. Si dan lo mismo, puedes confiar en el dato.

In [ ]:
import duckdb

SALIDA = "../datasets/salida"

publicado = duckdb.sql(
    f"SELECT ROUND(SUM(facturacion), 2) FROM '{SALIDA}/kpi_ciudad.csv'").fetchone()[0]
en_vivo = duckdb.sql("""SELECT ROUND(SUM(unidades*precio_unitario), 2)
                        FROM ventas_limpio WHERE ciudad IS NOT NULL""").fetchone()[0]

print(f"  publicado : {publicado}")
print(f"  en vivo   : {en_vivo}")
print("  COINCIDE" if abs(publicado - en_vivo) < 0.01
      else f"  NO COINCIDE. Diferencia: {abs(publicado - en_vivo)}")

## Lo que te llevas

| |
|---|
| Un cuadro de mando son **tres formas**: titular, ranking y evolución |
| `COPY … TO` convierte una consulta en **una pieza que otro consume** |
| El `IS NOT NULL` es **una decisión**, y las decisiones se declaran |
| Un formato de fecha con el año delante **ordena bien como texto** |
| ⚓ Publicar sin recontar no vale: **3 · 10 · 12** |

---

*Reto del LAB08 · solución comentada · Formación San Miguel · Zaragoza*